## Configuración para poder importar desde el src/*

In [9]:
import os
import sys
from pathlib import Path
ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [10]:
from dataclasses import dataclass
import json
from PIL import Image
from common.common_types import LayoutElement
from common.data_storage import DataStorage

@dataclass
class PageSample:
    images: list[Image.Image]
    elements: list[LayoutElement]

paths = DataStorage.find_json_paths()
dataset: list[PageSample] = []
for path in paths:
    with open(path) as f:
        data = json.load(f)
        images = DataStorage.get_images(path.stem)
        dataset.append(PageSample(images=images, elements=data))


In [11]:
all_labels = set()
for doc in dataset:
    for e in doc.elements:
        all_labels.add(e["label"])

label_list = sorted(list(all_labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Total facturas: {len(dataset)}")
print(f"Etiquetas: {label_list}")

Total facturas: 28
Etiquetas: ['FIELD_KEY_ADDRESS', 'FIELD_KEY_AMOUNT', 'FIELD_KEY_DATE', 'FIELD_KEY_EMAIL', 'FIELD_KEY_ID', 'FIELD_KEY_NAME', 'FIELD_KEY_TEXT', 'FIELD_VALUE_ADDRESS', 'FIELD_VALUE_AMOUNT', 'FIELD_VALUE_DATE', 'FIELD_VALUE_EMAIL', 'FIELD_VALUE_ID', 'FIELD_VALUE_NAME', 'FIELD_VALUE_TEXT', 'HEADER_PRODUCT_CODE', 'HEADER_PRODUCT_CODE_AUX', 'HEADER_PRODUCT_DETAIL', 'HEADER_PRODUCT_DISCOUNT', 'HEADER_PRODUCT_NAME', 'HEADER_PRODUCT_PRICE', 'HEADER_PRODUCT_QUANTITY', 'HEADER_PRODUCT_SUBSIDY', 'HEADER_PRODUCT_TOTAL', 'HEADER_PRODUCT_WITHOUT_SUBSIDY', 'ITEM_PRODUCT_CODE', 'ITEM_PRODUCT_CODE_AUX', 'ITEM_PRODUCT_DETAIL', 'ITEM_PRODUCT_DISCOUNT', 'ITEM_PRODUCT_NAME', 'ITEM_PRODUCT_PRICE', 'ITEM_PRODUCT_QUANTITY', 'ITEM_PRODUCT_SUBSIDY', 'ITEM_PRODUCT_TOTAL', 'ITEM_PRODUCT_WITHOUT_SUBSIDY', 'O']


## Preparación

In [12]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
from PIL import Image

def prepare_document(elements):
    words = [e["text"] for e in elements]
    boxes = [e["normalized_bbox"] for e in elements]
    labels = [label2id[e["label"]] for e in elements]
    return words, boxes, labels



def encode_document(image:Image.Image,elements: list[LayoutElement]):
    words, boxes, labels = prepare_document(elements)
  
    encoding = processor(
        images=image,
        text=words,
        boxes=boxes,
        word_labels=labels,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    return encoding




## Entrenamiento

In [14]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
import torch

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from torch.utils.data import Dataset as TorchDataset

def extract_per_page(page:PageSample):
    separated = []
    for index,image in enumerate(page.images):
        current_page = index + 1 
        current_elements = [e for e in page.elements if e["page"] == current_page]
        separated.append((image, current_elements))
    return separated

class InvoiceDataset(TorchDataset):
    def __init__(self, documents):
        self.documents = documents

    def __getitem__(self, idx):
        image, elements = self.documents[idx]
        encoding = encode_document(image, elements)
        return {k: v.squeeze(0) for k, v in encoding.items()}

    def __len__(self):
        return len(self.documents)
    
split = int(len(dataset) * 0.8)

train_data = dataset[:split]
eval_data = dataset[split:]

train_data_final = []

for page in train_data:
    train_data_final.extend(extract_per_page(page))

eval_data_final = []
for page in eval_data:
    eval_data_final.extend(extract_per_page(page))


train_dataset = InvoiceDataset(train_data_final)
val_dataset = InvoiceDataset(eval_data_final)

print()
print(f"Train: {len(train_data)} | Val: {len(eval_data)}")
print(f"Train pages: {len(train_data_final)} | Val pages: {len(eval_data_final)}")


Train: 22 | Val: 6
Train pages: 30 | Val pages: 11


In [16]:


training_args = TrainingArguments(
    output_dir="./model-output",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    save_steps=50,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

  0%|          | 0/150 [00:00<?, ?it/s]c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
  7%|▋         | 10/150 [00:05<01:22,  1.70it/s]

{'loss': 2.9459, 'grad_norm': 4.203189373016357, 'learning_rate': 4.666666666666667e-05, 'epoch': 0.67}


 10%|█         | 15/150 [00:09<01:16,  1.77it/s]

{'eval_loss': 2.2445485591888428, 'eval_runtime': 0.7372, 'eval_samples_per_second': 14.921, 'eval_steps_per_second': 8.139, 'epoch': 1.0}


c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 13%|█▎        | 20/150 [00:13<01:32,  1.41it/s]

...


100%|██████████| 150/150 [01:44<00:00,  1.80it/s]

{'eval_loss': 0.2615739107131958, 'eval_runtime': 0.7222, 'eval_samples_per_second': 15.231, 'eval_steps_per_second': 8.308, 'epoch': 10.0}


100%|██████████| 150/150 [01:46<00:00,  1.41it/s]

{'train_runtime': 106.038, 'train_samples_per_second': 2.829, 'train_steps_per_second': 1.415, 'train_loss': 0.6490126299858093, 'epoch': 10.0}


TrainOutput(global_step=150, training_loss=0.6490126299858093, metrics={'train_runtime': 106.038, 'train_samples_per_second': 2.829, 'train_steps_per_second': 1.415, 'total_flos': 79645736448000.0, 'train_loss': 0.6490126299858093, 'epoch': 10.0})